# VMware VM Socket Server (Jupyter Notebook Version)

此 Notebook 用於在 **VMware 虛擬機**上運行 Socket 伺服器。為了避免阻塞 Jupyter Kernel 導致無法執行其他 Cell，本版本使用**多執行緒 (Threading)** 將伺服器放在背景運行。

### 1. 啟動背景 Socket 伺服器 (Port 7777)

In [ ]:
import socket
import threading

# 控制伺服器開關的全局變數
server_running = True
server_thread = None

def run_server_threaded(host='0.0.0.0', port=7777):
    global server_running
    server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server_socket.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    # 設定 1 秒逾時，讓迴圈能定期檢查 server_running 狀態，以便安全退出
    server_socket.settimeout(1.0) 
    
    try:
        server_socket.bind((host, port))
        server_socket.listen(5)
        print(f"【伺服器已在背景啟動】監聽中 {host}:{port} ...")
        
        while server_running:
            try:
                client_socket, client_address = server_socket.accept()
                print(f"\n[+] 收到來自 {client_address} 的連線")
                
                data = client_socket.recv(1024)
                if data:
                    message = data.decode('utf-8')
                    print(f"[*] 收到訊息: '{message}'")
                    response = f"Hello Host! Connection is successful. Received: '{message}'"
                    client_socket.sendall(response.encode('utf-8'))
                client_socket.close()
            except socket.timeout:
                # 逾時是正常的，繼續迴圈以檢查 server_running
                continue
            except Exception as e:
                if server_running:
                    print(f"[!] 連線處理錯誤: {e}")
                break
    except Exception as e:
        print(f"[!] 伺服器啟動錯誤: {e}")
    finally:
        server_socket.close()
        print("\n【伺服器已安全關閉】")

# 啟動背景伺服器
server_running = True
server_thread = threading.Thread(target=run_server_threaded)
server_thread.start()

### 2. 停止背景伺服器
執行下方 Cell 來停止伺服器，釋放 Port `7777` 資源。

In [ ]:
server_running = False
if server_thread is not None:
    server_thread.join()
    print("伺服器執行緒已成功結束。")